## CS328 Assignment 2: Human Activity Recognition Using Machine Learning

The objective of this assignment is to develop your understanding and practical application skills in the field of machine learning, particularly focusing on Human Activity Recognition (HAR). Human Activity Recognition has become increasingly important in various fields including health care, personal fitness, smart homes, and surveillance. In this assignment, you will be working with the accelerometer data the class collected with Sensor Logger in the Activities Study, plus a small sample dataset recorded with a wrist-worn device. The data has been classified into seven distinct activities: walking, walking downstairs, running/jogging, lying, sitting, standing, and walking upstairs.

You will get hands-on experience in several key aspects of machine learning and data processing:
1. Data collection and preprocessing from multiple sources for analysis
2. Understand how to create windows of raw data through resampling and to extract meaningful features from these windows.
3. Hands-on experience on how to extract features, encode target variables, split datasets, train models, make predictions, and evaluate the performance of these models.
4. Learn to assess your trained model’s performance using common metrics like accuracy, confusion matrix and more. Understanding how well your models are performing is crucial in any machine learning task.

Through this assignment, you will be able to develop a pipeline for Human Activity Recognition that can be further fine-tuned for different tasks or datasets. You will be able to apply the skills and knowledge gained here to other machine learning projects and real-world applications.

#### Imports Block

Make sure all imports are in this block below (and leave the two comments IMPORTS START and IMPORT END as is. The extractor script uses the START and END delimitors when extracting the functions).

In [15]:
# -- IMPORTS START --
import pandas as pd
import glob
import re
import os
import sys
import pickle
import datetime
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime
from sklearn import tree, metrics
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from scipy.signal import butter, filtfilt, find_peaks
from sklearn.tree import DecisionTreeClassifier,export_graphviz
from sklearn.model_selection import train_test_split
from util import viz_tree, calc_magnitude, remove_noise, add_features
# -- IMPORTS END --

# enable zooming into graphs
%matplotlib inline
plt.rcParams['figure.figsize'] = [9, 6] # width, height in inches

#### About the data (read this before you start)

- **Class_Data** was recorded with Sensor Logger. Its `Accelerometer` sensor reports **linear acceleration in m/s² with gravity removed**: the phone combines the accelerometer and gyroscope to estimate the gravity vector and subtracts it from the raw signal. A phone lying still on a table reads ≈ 0 on all three axes, not 9.8.
- **Sample_Data** was recorded with a wrist-worn device. It has already been converted to match Class_Data (from *g* to m/s², and gravity removed), so both datasets are in the same units.
- Most phones log at ~100 Hz, but a few class recordings are at 50-60 Hz. Estimate the sampling rate of each file from its timestamps instead of hard-coding it.
- The column order differs between the two folders (`time,x,y,z` vs `time,seconds_elapsed,z,y,x`), so always select columns by name.

**Hint (why gravity matters):** with gravity removed, the orientation of the phone is lost. Lying, sitting, and standing all look like "almost no motion", so features computed on the acceleration magnitude will confuse them. Think about what is still different between these activities (e.g., how much the body sways while standing) and expect some confusion between these three classes. Walking, stairs, and jogging differ in intensity and rhythm, so frequency-domain and per-axis features are worth trying.

###  Assignment: Function 1: extract_features(data, window_sec, sample_rate, activity)

**Instructions:**
Define the `extract_features` function that extracts features from accelerometer data by applying a sliding window approach.


**Hints and Instructions:**
Use the `pandas.DataFrame.resample` function to implement the sliding window approach (it needs the time as index: `data.set_index('time').resample(f'{window_sec}s')`). Remember to use the computed features when appending to the new DataFrame.

- The function takes as arguments:
  - `data`: a DataFrame containing the `time` column, the raw `x`, `y`, `z` axes, and the filtered acceleration magnitude signal
  - `window_sec`: the window length in seconds for feature extraction
  - `sample_rate`: the sampling frequency of the accelerometer data
  - `activity`: the activity label for each window you extract
- The function will perform the following:
  - For each window in the resampled data:
    - Skip windows that are too short (e.g., fewer than 80% of `window_sec * sample_rate` samples). These are the partial windows at the start and end of a recording, or gaps in the logging.
    - Call the `add_my_features` function (Function 3) to compute features of the window
    - Add the `activity` label to the features
  - Append the computed features to a new DataFrame and return this DataFrame

In [7]:
# Function to extract windows and features
def extract_features(data, window_sec, sample_rate, activity):
    
    ######## TODO ###############
    dataFrame = pd.DataFrame(data)
    minSamples = int(0.8 * window_sec * sample_rate)
    frame = pd.DataFrame()

    for time in range(0, len(dataFrame), int(window_sec * sample_rate)):
        window = dataFrame.iloc[time:time + int(window_sec * sample_rate)]
        if len(window) >= minSamples:
            features = add_features(window)
            features['activity'] = activity
            frame = frame.append(features, ignore_index=True)
    #############################
    
    return frame

###  Assignment: Function 2: all_data_to_combined_csv(root, output_filename, window_sec)

This function is expected to process data collected from different activities, extract features and store everything into a combined CSV file.

**Hint and Instructions:**
Understand how the `glob`, `pandas` and `os` modules work. Understand how the `calc_magnitude` and `remove_noise` function work. The activity label is the name of the folder that contains the file, so use `os.path.basename(os.path.dirname(filename))` to extract the activity (file names such as `026.csv` or `accelerometer_<date>.csv` do not contain it).

- Write a function named `all_data_to_combined_csv(root, output_filename, window_sec)`.
- This function will process the data collected from various activities such as walking downstairs, jogging, lying, sitting, standing, walking upstairs, and walking. These activities' data are stored in different .csv files within their respective folders.
- the `root` will be the root folder of all the files.
- the `output_filename` will be the filename of the combined csv file.
- the `window_sec` will be the window length in seconds, so that you can try different window lengths.
- Use the glob module to create a list of all .csv files from each activity's directory.
- The pandas DataFrame `all_data` is initialized to store the processed data from all activities.
- The function will then loop over each activity's data file. For each file:
    - read each .csv file into a pandas DataFrame. `pd.read_csv` loads the `time` column as text, so parse it with `parse_dates=['time']` (this works once `transform_time_to_datetime` below has converted the Class_Data timestamps).
    - estimate the sampling rate of the file from its timestamps.
    - From the raw data, calculate the magnitude and remove any noise.
    - Extract the activity type for each file.
    - extract features using the `extract_features()` function.
    - add a `recording` column with the file name, so that windows can be grouped back into recordings later (majority vote and failure analysis).
    - append the feature frames into the all_data dataframe.
- Continue this process until all .csv files from all activities have been processed.
- Finally, the function writes the `all_data` DataFrame to `output_filename` and returns it.

In [8]:
def all_data_to_combined_csv(root, output_filename = 'all_data.csv', window_sec = 5):
    import os
    import sys

    ######## TODO ###############
    all_data_frames = []
    csv_files = glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)
    for file in csv_files:
        data = pd.read_csv(file)
        all_data_frames.append(data)
    all_data = pd.concat(all_data_frames, ignore_index=True)
    all_data.to_csv(output_filename, index=False)
    #############################

    return all_data

###  Assignment: Function 3: add_my_features(window, sample_rate)

This is where you do your **feature engineering** (exploration report §1). The function receives one window and returns a one-row DataFrame of features.

**Hints and Instructions:**
- Start from the baseline features in `add_features` (7 statistics of the filtered magnitude, from util.py) and add your own as new columns. As provided, the function only returns the baseline, so you can run the whole notebook before writing any features.
- Ideas to try:
  - Time domain: RMS, range, interquartile range, zero-crossing rate, number of peaks per second (`find_peaks`)
  - Frequency domain: dominant frequency and spectral energy in a band (use `np.fft.rfft` on the filtered magnitude and `np.fft.rfftfreq(len(signal), 1/sample_rate)`)
  - Per axis: standard deviation of `x`, `y`, `z`, or correlation between axes
- Aim for ~5-8 features in your final pipeline. Keep the ones that help and document the ones you dropped.

In [9]:
def add_my_features(window, sample_rate):
    features = add_features(window)   # baseline: avg, max, med, min, q25, q75, std of filtered_accel_mag

    ######## TODO ###############
    signal = window['filtered_accel_mag'].values
    
    # 1. Time-Domain: Root Mean Square (RMS) & Range
    features['rms'] = np.sqrt(np.mean(signal**2))
    features['range'] = np.ptp(signal)
    
    # 2. Time-Domain: Zero-Crossing Rate
    mean_centered = signal - np.mean(signal)
    zero_crossings = np.where(np.diff(np.signbit(mean_centered)))[0]
    features['zcr'] = len(zero_crossings) / (len(signal) / sample_rate)

    # 3. Time-Domain: Peak Frequency Rate via SciPy find_peaks
    peaks, _ = find_peaks(signal, prominence=0.3)
    features['peak_rate'] = len(peaks) / (len(signal) / sample_rate)

    # 4. Frequency-Domain: Dominant Frequency & Spectral Energy (RFFT)
    rfft_vals = np.abs(np.fft.rfft(mean_centered))
    rfft_freqs = np.fft.rfftfreq(len(mean_centered), d=1.0 / sample_rate)
    
    if len(rfft_vals) > 1:
        dom_idx = np.argmax(rfft_vals[1:]) + 1  # Exclude DC
        features['dom_freq'] = rfft_freqs[dom_idx]
        features['spectral_energy'] = np.sum(rfft_vals[1:]**2) / len(rfft_vals[1:])
    else:
        features['dom_freq'] = 0.0
        features['spectral_energy'] = 0.0

    # 5. Per-Axis Features: Individual Standard Deviations
    for axis in ['x', 'y', 'z']:
        if axis in window.columns:
            features[f'std_{axis}'] = window[axis].std()
        else:
            features[f'std_{axis}'] = 0.0
    #############################

    return features

## Assignment: Transform time to datetime


Due to the fact that sensor logger data has a UTC time in nanoseconds instead of datetime, we should transform the time first. Feel free to modify the below function so that all the collected data has the same datetime format as the provided data. Check the Sample_Data folder to see the format we need.

Note that each activity data is in its own folder. For example, the collected downstairs data is in `data/Class_Data/downstairs`.

In [10]:
def transform_time_to_datetime(root):
    # Get list of all activity folders
    activity_folders = [f for f in os.listdir(root) if os.path.isdir(os.path.join(root, f))]

    for folder in activity_folders:
        files = glob.glob(f"{root}/{folder}/*.csv")
        for filename in files:
            df = pd.read_csv(filename)
            # Sensor Logger time is nanoseconds since 1970 (UTC); skip files that are already converted
            if pd.api.types.is_numeric_dtype(df['time']):
                df['time'] = pd.to_datetime(df['time'], unit='ns')
                df.to_csv(filename, index=False)

In [11]:
transform_time_to_datetime('./data/Class_Data')

#### Helper functions (provided)

`train_decision_tree` and `evaluate` take an optional `features` argument (a list of column names). By default, `train_decision_tree` uses every feature column, i.e., everything except `activity` and `recording`, and `evaluate` uses the same columns the model was trained on.

In [12]:
BASELINE_FEATURES = ['avg', 'max', 'med', 'min', 'q25', 'q75', 'std']

def feature_columns(frames):
    return [c for c in frames.columns if c not in ('activity', 'recording')]


def evaluate(model, frames, features=None):
    # By default, use the same columns the model was trained on
    if features is None:
        features = getattr(model, 'feature_names_in_', feature_columns(frames))
    X_test = frames[list(features)]

    # Extract target column
    y_test = frames['activity']

    pred = model.predict(X_test)
    # Evaluate on test set
    acc = model.score(X_test, y_test)
    print(classification_report(y_test, pred, zero_division=0))
    print("Accuracy on test set:", acc)
    return acc, pred

In [13]:
def train_decision_tree(frames, max_depth=5, criterion='entropy', features=None):
    # Extract feature columns
    if features is None:
        features = feature_columns(frames)
    X = frames[list(features)]

    # Extract target column
    y = frames['activity']

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # Create model
    dt_model = DecisionTreeClassifier(criterion=criterion, max_depth=max_depth, random_state=42).fit(X_train, y_train)
    dt_pred = dt_model.predict(X_test)

    # Evaluate on test set
    acc = dt_model.score(X_test, y_test)
    dt_cm = confusion_matrix(y_test, dt_pred, labels=dt_model.classes_)
    print(classification_report(y_test, dt_pred, zero_division=0))
    print("Accuracy on test set:", acc)

    return dt_model,dt_cm,acc

## Assignment-Feature Exploration: Window Length and Features (Class_Data)

Before comparing models, choose the window length and the feature set (the **feature engineering** axis in RUBRIC.md, exploration report §1). Use only the class corpus and the 30% test split that `train_decision_tree` returns, with its default hyperparameters.

1. **Window length**: Build the class corpus with at least three window lengths (e.g., 2 s, 5 s, 10 s) and train `train_decision_tree` on each, using only the baseline features (`features=BASELINE_FEATURES`).

2. **Features**: Write `add_my_features` (Function 3) and rebuild the class corpus. Compare the baseline features to your features at the same window length. Try adding and removing features (pass `features=[...]` to `train_decision_tree`) and use the classification report to see which classes each feature helps.

3. **Lock in** one window length and one feature set, and use them in all the sections below.

4. **Fill up** this table.

5. **Interpretation of the results**: Which window length worked best, and why do you think so? Which features helped, and which class pairs did they separate? Which classes are still confused?

In [14]:
# Hints, step by step:
# Build the class corpus with different window lengths, e.g. all_data_to_combined_csv('./data/Class_Data', 'class_2s.csv', window_sec=2)
# Train train_decision_tree on each with features=BASELINE_FEATURES and compare the accuracy
# Write add_my_features, rebuild, and compare baseline vs your features at the same window length
# Lock in WINDOW_SEC and FEATURES for the rest of the notebook

########################### TODO ##########################
class_2s = all_data_to_combined_csv('./data/Class_Data', 'class_2s.csv', window_sec=2)
class_5s = all_data_to_combined_csv('./data/Class_Data', 'class_5s.csv', window_sec=5)
class_10s = all_data_to_combined_csv('./data/Class_Data', 'class_10s.csv', window_sec=10)

dt_2s, _, acc_2s = train_decision_tree(class_2s, features=BASELINE_FEATURES)
dt_5s, _, acc_5s = train_decision_tree(class_5s, features=BASELINE_FEATURES)
dt_10s, _, acc_10s = train_decision_tree(class_10s, features=BASELINE_FEATURES)

MY_FEATURES = ['avg', 'std', 'rms', 'range', 'dom_freq', 'spectral_energy', 'std_x', 'std_y', 'std_z']
dt_my, _, acc_my = train_decision_tree(class_5s, features=MY_FEATURES)

KeyError: "None of [Index(['avg', 'max', 'med', 'min', 'q25', 'q75', 'std'], dtype='str')] are in the [columns]"

**For the data in the dataset:**

| Window length | Features | Accuracy (Class_Data test split) |
|-|-|-|
| 2 s | baseline | **FILL UP** |
| 5 s | baseline | **FILL UP** |
| 10 s | baseline | **FILL UP** |
| **FILL UP** s | **FILL UP your features** | **FILL UP** |


#### 🛑 Interpretation of the results

**Write Below:**

## Assignment-Classification1: Testing Decision Tree Classifier Performance (Train: Sample_Data, Test: Class_Data)

In this section, we will evaluate the overall performance of our activity classifier using a combined dataset of all activities. This process helps understand how accurately the model can classify a wide variety of activities. The steps are as follows:

1. **Data Generation**: We start by calling the `all_data_to_combined_csv()` function with your chosen window length, for both Sample_Data and Class_Data. This function processes all the individual activity datasets, applies the necessary preprocessing, and generates a combined CSV file. If you change your preprocessing or features, call it again to rebuild the file.

2. **Data Loading**: Once we have the combined CSV file, we load it into a pandas DataFrame for further use (or use the DataFrame the function returns).

3. **Model Training**: The `train_decision_tree` function is then called. This function trains the decision tree model and evaluates its performance, printing the precision, recall, and accuracy metrics. It also returns the trained model, the confusion matrix, and the overall accuracy for further use. Train a model using sample data and evaluate it on the class corpus. Use the default hyper parameters in `train_decision_tree`.

4. **Grid Search**: Do grid search for tuning the hyperparameters (max_depth and criterion in `train_decision_tree`). Basically train models using sample data with different hyperparameters and pick the hyperparameters with the highest accuracy on the test split of the sample data (`train_decision_tree` returns it). Then evaluate the best model on the class corpus. Do not pick hyperparameters by looking at the class corpus accuracy: it is your test set. Finally report accuracy and hyperparameters as well.

5. **Performance Visualization**: To provide a visual understanding of the model's performance, we display the confusion matrix using matplotlib's `ConfusionMatrixDisplay` function. Each row of the matrix corresponds to a true class, while each column corresponds to a predicted class. The diagonal elements represent correctly classified instances, and off-diagonal elements are instances that are misclassified. Visualize the confusion matrix for the test set.

6. **Decision Tree Visualization**: Finally, visualize the decision tree using the `viz_tree` function (loaded from util.py). This function generates a graphic representation of the decision tree model (the top levels), showing how it makes decisions based on the features.

7. **Fill up** this table based on the results you obtain by trying different parameters (fill both rows). We will evaluate the accuracy of numbers you fill up below.

8. **Interpretation of the results**: What do these accuracy scores suggest about the ability of the model to distinguish between these activities? Do some activities appear to be more distinguishable than others? what did you learn from hyperparameter tuning? Remember to provide a brief discussion for each point.

This section's output can help understand how well the model generalizes to different activities and how different features influence the model's decisions. Also, you will see the effect of hyperparameter tuning. Remember that it's normal if the model performs better on some activities than others, depending on the complexity and distinctiveness of the activities.

In [ ]:
# Hints, step by step:
# Combine all the data under data/Sample_Data
# Combine all the data under data/Class_Data
# Train the decision tree
# Grid search, choosing by the Sample_Data test split accuracy
# Display the confusion matrix
# Visualize the tree
# Evaluate the model on class dataset

########################### TODO ##########################
# Execution Code
sample_data = all_data_to_combined_csv('./data/Sample_Data', 'sample_5s.csv', window_sec=5)
class_data = all_data_to_combined_csv('./data/Class_Data', 'class_5s.csv', window_sec=5)

# 1. Default
model_def, _, acc_sample_def = train_decision_tree(sample_data, max_depth=5, criterion='entropy', features=MY_FEATURES)
acc_class_def, _ = evaluate(model_def, class_data, features=MY_FEATURES)

# 2. Grid Search on Sample_Data
best_depth, best_crit, best_acc = 5, 'entropy', 0.0
for d in [3, 5, 7, 10, None]:
    for c in ['gini', 'entropy']:
        m, _, acc = train_decision_tree(sample_data, max_depth=d, criterion=c, features=MY_FEATURES)
        if acc > best_acc:
            best_acc, best_depth, best_crit = acc, d, c

best_model_c1, _, acc_sample_best = train_decision_tree(sample_data, max_depth=best_depth, criterion=best_crit, features=MY_FEATURES)
acc_class_best, _ = evaluate(best_model_c1, class_data, features=MY_FEATURES)

**For the data in the dataset:**

| Hyperparameters | Accuracy (Sample_Data test split) | Accuracy (Class_Data) |
|-|-|-|
| Default (max_depth=5, criterion='entropy') | **FILL UP** | **FILL UP** |
| **FILL UP best hyperparameters**  | **FILL UP** | **FILL UP** |


#### 🛑 Interpretation of the results

**Write Below:**

## Assignment-Classification2: Testing Decision Tree Classifier Performance (Train: Class_Data, Test: Sample_Data)

In this section, you will do a grid search for hyperparameter tuning using class dataset. When you found the best model, you will evaluate it on the sample data. Check the instruction below for this section:

1. **Grid Search**: This time only use the class corpus for tuning the hyperparameters using grid search (max_depth and criterion in `train_decision_tree`). Basically train and evaluate models with different hyperparameters only using the class dataset and see what hyperparameters results in better performance on test split of class dataset (`train_decision_tree` returns it). Finally evaluate the best model on the sample dataset. 


2. **Performance Visualization**: To provide a visual understanding of the model's performance, we display the confusion matrix using matplotlib's `ConfusionMatrixDisplay` function. Each row of the matrix corresponds to a true class, while each column corresponds to a predicted class. The diagonal elements represent correctly classified instances, and off-diagonal elements are instances that are misclassified. Visualize the confusion matrix for the test set.

3. **Decision Tree Visualization**: Finally, visualize the decision tree using the `viz_tree` function (loaded from util.py). This function generates a graphic representation of the decision tree model, showing how it makes decisions based on the features.

4. **Fill up** this table based on the results you obtain by trying different parameters (fill both rows). We will evaluate the accuracy of numbers you fill up below.

5. **Interpretation of the results**: What do these accuracy scores suggest about the ability of the model to distinguish between these activities? Do some activities appear to be more distinguishable than others? what did you learn from hyperparameter tuning? Remember to provide a brief discussion for each point.

In [ ]:
# roughly similar flow as previous section

########################### TODO ##########################
from sklearn.ensemble import RandomForestClassifier

# Pipeline execution with Random Forest
rf_sample = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(sample_data[MY_FEATURES], sample_data['activity'], test_size=0.3, random_state=42)
rf_sample.fit(X_train, y_train)

acc_rf_sample_split = rf_sample.score(X_test, y_test)
acc_rf_class_eval, _ = evaluate(rf_sample, class_data, features=MY_FEATURES)

**For the data in the dataset:**

| Hyperparameters | Accuracy (Class_Data test split) | Accuracy (Sample_Data) |
|-|-|-|
| Default (max_depth=5, criterion='entropy') | **FILL UP** | **FILL UP** |
| **FILL UP best hyperparameters**  | **FILL UP** | **FILL UP** |


#### 🛑 Interpretation of the results

**Write Below:**

## Assignment-Classification3: Testing a new Classifier Performance (Train: Sample_Data, Test: Class_Data)

Repeat section Assignment-Classification1 with a different classifier, using the same window length and feature set. It could be Random Forest, kNN, Logistic regression, or any other classifier of your choice (add its import to the imports block). Tune its key hyperparameters the same way as in Classification1. Finally compare it with the decision tree in the assignment-classification1 section (model exploration, exploration report §2).

Note: kNN, Logistic Regression, and SVM are sensitive to feature scale, so standardize the features first (e.g., `make_pipeline(StandardScaler(), KNeighborsClassifier())`).

In [ ]:
# Hints, step by step:
# Write a code for evaluation and training the new classifier
# Train the new classifier
# Display the confusion matrix
# Visualize the model if it makes sense (a tree, or feature importances for a forest)
# Evaluate the model on class dataset

########################### TODO ##########################

**For the data in the dataset:**

| Hyperparameters | Accuracy (Sample_Data test split) | Accuracy (Class_Data) |
|-|-|-|
| **FILL UP best hyperparameters**  | **FILL UP** | **FILL UP** |


#### 🛑 Interpretation of the results and comparison to the decision tree.

**Write Below:**

## Assignment-Classification4: Testing a new Classifier Performance (Train: Class_Data, Test: Sample_Data)

Repeat section Assignment-Classification2 with your new classifier. Also, compare it with the decision tree in the assignment-classification2 section. Did the better model help all classes equally, or did the gain concentrate on specific classes?

In [ ]:
# roughly similar flow as previous section

########################### TODO ##########################

**For the data in the dataset:**

| Hyperparameters | Accuracy (Class_Data test split) | Accuracy (Sample_Data) |
|-|-|-|
| **FILL UP best hyperparameters**  | **FILL UP** | **FILL UP** |


#### 🛑 Interpretation of the results and comparison to the decision tree.

**Write Below:**

###  Assignment: Function 4: predict_recordings(model, frames, features)

So far every number is a **window** accuracy. A real app labels a whole recording, so aggregate the window predictions of each recording into one label (exploration report §3).

**Hints and Instructions:**
- The function takes as arguments:
  - `model`: a trained classifier
  - `frames`: a feature DataFrame with the `activity` and `recording` columns
  - `features`: the feature columns the model was trained on
- The function will perform the following:
  - Predict the label of every window
  - Group the predictions by `recording` and take the most common label (majority vote). You may also try confidence-weighting with `model.predict_proba`.
  - Return a DataFrame with one row per recording and the columns `true` and `predicted`

In [ ]:
def predict_recordings(model, frames, features=None):
    # By default, use the same columns the model was trained on
    if features is None:
        features = getattr(model, 'feature_names_in_', feature_columns(frames))

    ######## TODO ###############
    pass
    #############################

    return recordings

## Assignment-Classification5: Recording-Level Predictions

Use `predict_recordings` with your best decision tree and your best new classifier from Classification2 and Classification4 (trained on Class_Data), on Sample_Data. Then do the same for your models from Classification1 and Classification3 (trained on Sample_Data), on Class_Data.

1. **Fill up** this table.

2. **Interpretation of the results**: Does aggregating help? For which classes? When can majority vote make things worse?

In [ ]:
# Hints, step by step:
# Call predict_recordings for each model and test set
# Recording-level accuracy = (recordings['true'] == recordings['predicted']).mean()
# Display a confusion matrix of the recording-level predictions

########################### TODO ##########################
# Execution Code
rec_dt_c2s = predict_recordings(best_model_c2, sample_data, features=MY_FEATURES)
rec_rf_c2s = predict_recordings(rf_class_model, sample_data, features=MY_FEATURES)

rec_dt_s2c = predict_recordings(best_model_c1, class_data, features=MY_FEATURES)
rec_rf_s2c = predict_recordings(rf_sample_model, class_data, features=MY_FEATURES)

acc_dt_c2s = (rec_dt_c2s['true'] == rec_dt_c2s['predicted']).mean()
acc_rf_c2s = (rec_rf_c2s['true'] == rec_rf_c2s['predicted']).mean()
acc_dt_s2c = (rec_dt_s2c['true'] == rec_dt_s2c['predicted']).mean()
acc_rf_s2c = (rec_rf_s2c['true'] == rec_rf_s2c['predicted']).mean()

**For the data in the dataset:**

| Model | Train → Test | Window accuracy | Recording accuracy |
|-|-|-|-|
| Decision Tree (best) | Class_Data → Sample_Data | **FILL UP** | **FILL UP** |
| **FILL UP your new classifier** (best) | Class_Data → Sample_Data | **FILL UP** | **FILL UP** |
| Decision Tree (best) | Sample_Data → Class_Data | **FILL UP** | **FILL UP** |
| **FILL UP your new classifier** (best) | Sample_Data → Class_Data | **FILL UP** | **FILL UP** |


#### 🛑 Interpretation of the results

**Write Below:**

## Assignment-Failure Analysis (for exploration report §4 and §5)

Pick your final pipeline (exploration report §3) and look at where it fails.

1. From its confusion matrix, find the **top 2 failure modes** (the largest off-diagonal cells, e.g., upstairs predicted as walk).
2. For each failure mode, use the `recording` column to find a recording or window where the failure is clear. Plot its trace (`filtered_accel_mag` over time) and compare its feature values to the typical (median) feature values of the correct class and of the predicted class.
3. Use these figures in §5 of your exploration report, and decide whether each failure is a **feature** problem or a **model** problem.

In [ ]:
# Hints, step by step:
# Find misclassified windows: frames[pred != frames['activity']]
# Pick one recording, reload its CSV, apply calc_magnitude and remove_noise, and plot filtered_accel_mag
# Compare: frames.groupby('activity')[FEATURES].median() vs the feature values of the failing window

########################### TODO ##########################